# Baseline — imports

Charge les composants du modèle et de l’agent.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.model import load_model
from src.agents.baseline import BaselineAgent

# Chargement du modèle

Initialise Qwen, le tokenizer et le device.

In [2]:
llm = load_model()

Chargement du modèle : Qwen/Qwen2.5-0.5B-Instruct
Device utilisé : cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

# Données de test

Définit les règles et une fiche fictive.

In [3]:
rules = """
Une personne est admissible si elle a au moins 18 ans.
"""

fiche = """
Nom : Alice
Âge : 22 ans
"""

# Test de l’agent

Génère une réponse et extrait le verdict.

In [4]:
agent = BaselineAgent(
    model=llm.model,
    tokenizer=llm.tokenizer,
    device=llm.device,
    rules=rules,
    fiche=fiche,
)

result = agent.ask(
    "Alice est-elle admissible ?"
)

print(result["response"])
print()
print("Verdict extrait :", result["verdict"])

[VERDICT: ADMISSIBLE]

Verdict extrait : ADMISSIBLE


# Chat template

Vérifie le format conversationnel envoyé à Qwen.

In [5]:
messages = [
    {
        "role": "system",
        "content": agent.system_prompt,
    },
    {
        "role": "user",
        "content": "Alice est-elle admissible ?",
    },
]

formatted = llm.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(formatted)

<|im_start|>system
Tu dois prendre une décision uniquement à partir des règles et de la fiche.

RÈGLES :

Une personne est admissible si elle a au moins 18 ans.


FICHE :

Nom : Alice
Âge : 22 ans


Contraintes obligatoires :
- N'invente aucune information.
- Applique explicitement les règles fournies.
- Réponds brièvement.
- Termine TOUJOURS par exactement une des lignes suivantes :

[VERDICT: ADMISSIBLE]
[VERDICT: NON_ADMISSIBLE]
[VERDICT: INDETERMINE]

N'utilise jamais [VRAI], [FAUX], [OUI] ou [NON].<|im_end|>
<|im_start|>user
Alice est-elle admissible ?<|im_end|>
<|im_start|>assistant



# Historique borné

Vérifie que seuls les derniers tours sont conservés.

In [6]:
agent.ask("Quel âge a Alice ?")
agent.ask("Quelle règle as-tu utilisée ?")
agent.ask("Rappelle-moi ta décision.")
agent.ask("Explique-la encore une fois.")

history = agent.history.get_messages()

print("Nombre de messages :", len(history))

for message in history:
    print(
        message["role"],
        ":",
        message["content"][:100],
    )

Nombre de messages : 6
user : Quelle règle as-tu utilisée ?
assistant : Je n'ai pas utilisé de règle spécifique dans cette question, mais je suis basé sur les règles donnée
user : Rappelle-moi ta décision.
assistant : [VERDICT: ADMISSIBLE]
user : Explique-la encore une fois.
assistant : En tant qu'intelligence artificielle, je ne peux pas avoir d'âge ni être capable de faire de l'âge. 
